In [ ]:
import mlflow
import mlflow.sklearn
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
from sklearn.datasets import load_diabetes

mlflow.set_tracking_uri("http://localhost:5000")

mlflow.set_experiment("Diabetes_Prediction_Experiment")

data = load_diabetes()
X_train, X_test, y_train, y_test = train_test_split(
    data.data, data.target, test_size=0.2, random_state=42
)

with mlflow.start_run(run_name="LinearRegression_Run") as run:
    model = LinearRegression()
    model.fit(X_train, y_train)

    preds = model.predict(X_test)
    mse = mean_squared_error(y_test, preds)

    mlflow.log_param("model_type", "LinearRegression")
    mlflow.log_metric("mse", mse)
    mlflow.sklearn.log_model(model, artifact_path="model")

    print(f"Model logged in run: {run.info.run_id}")


2025/11/06 23:41:12 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2025/11/06 23:41:15 WARNING mlflow.models.model: Model logged without a signature and input example. Please set `input_example` parameter when logging the model to auto infer the model signature.


Model logged in run: 056efcbe7612418f94dfd8b952738076
🏃 View run LinearRegression_Run at: http://localhost:5000/#/experiments/1/runs/056efcbe7612418f94dfd8b952738076
🧪 View experiment at: http://localhost:5000/#/experiments/1


In [8]:
model_uri = f"runs:/{run.info.run_id}/model"
model_name = "Diabetes_Regression_Model"

registered_model = mlflow.register_model(model_uri=model_uri, name=model_name)
print("Model registered as:", registered_model.name)

Registered model 'Diabetes_Regression_Model' already exists. Creating a new version of this model...
2025/11/07 00:40:29 WARNING mlflow.tracking._model_registry.fluent: Run with id 056efcbe7612418f94dfd8b952738076 has no artifacts at artifact path 'model', registering model based on models:/m-82006c1b145b476b87669f6b3a632972 instead
2025/11/07 00:40:29 INFO mlflow.store.model_registry.abstract_store: Waiting up to 300 seconds for model version to finish creation. Model name: Diabetes_Regression_Model, version 3


Model registered as: Diabetes_Regression_Model


Created version '3' of model 'Diabetes_Regression_Model'.


In [9]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

latest_version = client.get_latest_versions(model_name, stages=["None"])[0].version

client.transition_model_version_stage(
    name=model_name,
    version=latest_version,
    stage="Staging"
)


C:\Temp\ipykernel_2092\851196374.py:5: FutureWarning: ``mlflow.tracking.client.MlflowClient.get_latest_versions`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  latest_version = client.get_latest_versions(model_name, stages=["None"])[0].version
C:\Temp\ipykernel_2092\851196374.py:7: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1762456229189, current_stage='Staging', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1762456280876, metrics=None, model_id=None, name='Diabetes_Regression_Model', params=None, run_id='056efcbe7612418f94dfd8b952738076', run_link='', source='models:/m-82006c1b145b476b87669f6b3a632972', status='READY', status_message=None, tags={}, user_id='', version='3'>

In [ ]:
client.transition_model_version_stage(
    name=model_name,
    version=latest_version,
    stage="Production",
)


C:\Temp\ipykernel_2092\4236177729.py:1: FutureWarning: ``mlflow.tracking.client.MlflowClient.transition_model_version_stage`` is deprecated since 2.9.0. Model registry stages will be removed in a future major release. To learn more about the deprecation of model registry stages, see our migration guide here: https://mlflow.org/docs/latest/model-registry.html#migrating-from-stages
  client.transition_model_version_stage(


<ModelVersion: aliases=[], creation_timestamp=1762456229189, current_stage='Production', deployment_job_state=<ModelVersionDeploymentJobState: current_task_name='', job_id='', job_state='DEPLOYMENT_JOB_CONNECTION_STATE_UNSPECIFIED', run_id='', run_state='DEPLOYMENT_JOB_RUN_STATE_UNSPECIFIED'>, description='', last_updated_timestamp=1762456333931, metrics=None, model_id=None, name='Diabetes_Regression_Model', params=None, run_id='056efcbe7612418f94dfd8b952738076', run_link='', source='models:/m-82006c1b145b476b87669f6b3a632972', status='READY', status_message=None, tags={}, user_id='', version='3'>

In [5]:
model = mlflow.pyfunc.load_model(model_uri="models:/Diabetes_Regression_Model/Production")

preds = model.predict(X_test)
print(preds[:5])

[139.5475584  179.51720835 134.03875572 291.41702925 123.78965872]
